## Import

In [1]:
import os
import numpy as np
import pandas as pd
import math
import pickle
from PIL import Image
from box import Box

import torch
from torch import nn
import torch.utils.data as data
from torch.utils.data import Dataset, DataLoader, Subset
from torch.nn import functional as F

import torchvision
from torchvision import transforms as T
from torchvision.io import read_image

import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint

from sklearn.model_selection import StratifiedKFold, train_test_split

from fastai.vision.all import BCEWithLogitsLossFlat

from timm import create_model


## Config

In [2]:
args = {'folder_name': '../input/petfinder-pawpularity-score',
    'seed': 1212,
    # 'num_folds': 5,
    'batch_size': 16,
    'num_workers': 8,
    'imagesize': 384,
    'valid_ratio': .2,
    'num_epoches': 5,
    'lr': 1e-4,
    'model_name': 'swin_large_patch4_window12_384'}

In [3]:
def fix_random_seed(seed_value):
    """the function to fix random seed of torch, numpy, cuda and cudnn
    Args:
        seed_value (int): seed of random
    Returns:
        None
    """
    # random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True
        

fix_random_seed(args['seed'])

In [4]:
# def train_val_split(dataset, val_ratio=0.2, seed=None):
#     """The function can split the dataset to train dataset and validation dataset by val_ratio.
#     Args:
#         dataset (class): The entire dataset which you want to split.
#         val_split (float): The ratio of validation dataset, and the value is [0., 1.]
#         seed (int): the random seed to fix the sklearn random state. -> None
#     Returns:
#         train_dataset
#         val_dataset
#     """
#     if seed is not None:
#         train_idx, val_idx = train_test_split(list(range(len(dataset))), test_size=val_ratio, random_state=seed)
#     else:
#         train_idx, val_idx = train_test_split(list(range(len(dataset))), test_size=val_ratio)
        
#     return Subset(dataset, train_idx), Subset(dataset, val_idx)

def train_valid_split(df, val_ratio=0.2, seed=None):
    """將DataFrame隨機劃分為train和valid，並返回train和valid的DataFrame"""
    if seed is not None:
        np.random.seed(seed)

    # 隨機抽樣，將索引劃分為train和valid
    indices = np.random.permutation(df.index)
    val_size = int(len(df) * val_ratio)
    val_indices = indices[:val_size]
    train_indices = indices[val_size:]

    # 根據索引劃分train和valid
    train_df = df.loc[train_indices]
    valid_df = df.loc[val_indices]
    return train_df, valid_df

## Dataset

In [5]:
class CustomImageDataset(Dataset):
    def __init__(self, df, transform=None):
        self.file_list = df['filename'].to_list()
        self.label_list = df['norm_score'].to_list()
        self.transform = transform
        
    def __len__(self):
        return len(self.label_list)

    def __getitem__(self, idx):
        img_path = self.file_list[idx]
        image = Image.open(img_path)
        
        label = self.label_list[idx]
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

In [6]:
class DataModule(pl.LightningDataModule):
    def __init__(self, args):
        super().__init__()
        self.args = args
        
        self.train_transform =  T.Compose([T.Resize((args['imagesize'],args['imagesize'])),
                                    T.RandomApply([T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1)], p=0.5),
                                    T.RandomHorizontalFlip(p=0.5),
                                    T.RandomVerticalFlip(p=0.5),
                                    T.RandomRotation(degrees=15, fill=0),
                                    T.RandomAffine(degrees=0, translate=(0.2, 0.2), shear=0.2),
                                    # T.RandomErasing(p=0.3, scale=(0.02, 0.1), ratio=(0.3, 3.3), value=0, inplace=False),
                                    T.ToTensor()
                                ])
        
        self.valid_transform =  T.Compose([T.Resize((args['imagesize'],args['imagesize'])),
                                    T.ToTensor()
                                ])
    
    def setup(self, stage=None):
        # First we read in our training data and create a new column with the image file locations
        df = pd.read_csv(os.path.join(args['folder_name'],"train.csv"))
        df['filename'] = df['Id'].map(lambda x:str(os.path.join(args['folder_name'],'train',x))+'.jpg')

        df = df.drop(columns=['Id']) # Drop ID column and reset the index
        df['norm_score'] = df['Pawpularity']/100 # Mapping the score to [0, 1]
        
        # 隨機抽樣，將索引劃分為train和valid
        indices = np.random.permutation(df.index)
        val_size = int(len(df) * self.args['valid_ratio'])
        val_indices = indices[:val_size]
        train_indices = indices[val_size:]

        # 根據索引劃分train和valid
        self.train_df = df.loc[train_indices]
        self.valid_df = df.loc[val_indices]
        
        self.train = CustomImageDataset(self.train_df, transform=self.train_transform)
        self.valid = CustomImageDataset(self.valid_df, transform=self.valid_transform)
        
    def train_dataloader(self):
        return DataLoader(self.train, batch_size=args['batch_size'], shuffle=True, num_workers=args['num_workers'], pin_memory=True)
    
    def val_dataloader(self):
        return DataLoader(self.valid, batch_size=args['batch_size'], shuffle=False, num_workers=args['num_workers'], pin_memory=True)
    
    def test_dataloader(self):
        return None


## Model

In [7]:
class PawpularityModule(pl.LightningModule):
    def __init__(self, args):
        super().__init__()
        def rmse(input,target):
            return 100*torch.sqrt(F.mse_loss(torch.sigmoid(input.flatten()), target))
        
        self.args = args
        self.loss_fn = BCEWithLogitsLossFlat()
        self.metric_fn = rmse
        
        model = create_model(self.args['model_name'], pretrained=True, num_classes=1)
        model = torch.compile(model) # Compile the model to boost
        self.model = model
        
        self.save_hyperparameters(args) # Save the hyparameters to hparams.yaml
        
    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self.model(x)
        loss = self.loss_fn(y_hat, y)
        self.log('train_loss', loss, prog_bar=True)
        return loss
    
    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self.model(x)
        loss = self.loss_fn(y_hat, y)
        metric = self.metric_fn(y_hat.sigmoid(), y)
        self.log('valid_loss', loss, prog_bar=True)
        self.log('valid_metric', metric, prog_bar=True)
        return metric
    
    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.args['lr'])
        return optimizer


## Train

In [9]:
# saves top-K checkpoints based on "val_loss" metric
checkpoint_callback = ModelCheckpoint(
    save_top_k=10,
    monitor="train_loss",
    mode="min",
    filename="{epoch:02d}-{valid_loss:.2e}-{val_metric:.3f}", # Don't add the metric_name (auto)
    auto_insert_metric_name = True, # If you want to set the metric name by yourself, please the auto_insert_metric_name=False
    # save_on_train_epoch_end=False,
)

In [9]:
model = PawpularityModule(args)

# The value is decided by your hardware, disable this line and it will tell you the best choicem. (3090 -> high)
torch.set_float32_matmul_precision('high') 

# benchmark: boost the cuda
# "16-mixed": for Automatic Mixed Precision (AMP), it can accelerate training and decrease the usage fo vRAM
trainer = pl.Trainer(max_epochs=args['num_epoches'], benchmark=True, callbacks=[checkpoint_callback], precision='16-mixed')

# If your dataloader is LightningDataModule, please use "datamodule".
# Or use "train_dataloaders" and "valid_dataloaders"
trainer.fit(model, datamodule=DataModule(args))

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type            | Params
------------------------------------------
0 | model | OptimizedModule | 195 M 
------------------------------------------
195 M     Trainable params
0         Non-trainable params
195 M     Total params
780.800   Total estimated model params size (MB)


Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

`Trainer.fit` stopped: `max_epochs=5` reached.


## Test

In [10]:
model = PawpularityModule.load_from_checkpoint(checkpoint_path="./lightning_logs/version_10/checkpoints/epoch=02-valid_loss=0.00e+00-val_metric=27.594.ckpt")

# The value is decided by your hardware, disable this line and it will tell you the best choicem. (3090 -> high)
torch.set_float32_matmul_precision('high') 

trainer = pl.Trainer(max_epochs=args['num_epoches'], benchmark=True, callbacks=[checkpoint_callback], precision='16-mixed')
trainer.validate(model, datamodule=DataModule(args))

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: 0it [00:00, ?it/s]

Failed to collect metadata on function, produced code may be suboptimal.  Known situations this can occur are inference mode only compilation involving resize_ or prims (!schema.hasAnyAliasInfo() INTERNAL ASSERT FAILED); if your situation looks different please file a bug to PyTorch.
Traceback (most recent call last):
  File "/home/rxchi1d/anaconda3/envs/ML2/lib/python3.9/site-packages/torch/_functorch/aot_autograd.py", line 1674, in aot_wrapper_dedupe
    fw_metadata, _out = run_functionalized_fw_and_collect_metadata(
  File "/home/rxchi1d/anaconda3/envs/ML2/lib/python3.9/site-packages/torch/_functorch/aot_autograd.py", line 606, in inner
    flat_f_outs = f(*flat_f_args)
  File "/home/rxchi1d/anaconda3/envs/ML2/lib/python3.9/site-packages/torch/_functorch/aot_autograd.py", line 2776, in functional_call
    out = Interpreter(mod).run(*args[params_len:], **kwargs)
  File "/home/rxchi1d/anaconda3/envs/ML2/lib/python3.9/site-packages/torch/fx/interpreter.py", line 136, in run
    self.en

BackendCompilerFailed: debug_wrapper raised RuntimeError: Inference tensors do not track version counter.

While executing %view_4 : [#users=1] = call_method[target=view](args = (%self_layers_0_blocks_0_attn_relative_position_index, -1), kwargs = {})
Original traceback:
  File "/home/rxchi1d/anaconda3/envs/ML2/lib/python3.9/site-packages/timm/models/swin_transformer.py", line 188, in _get_rel_pos_bias
    self.relative_position_index.view(-1)].view(self.window_area, self.window_area, -1)  # Wh*Ww,Wh*Ww,nH
 |   File "/home/rxchi1d/anaconda3/envs/ML2/lib/python3.9/site-packages/timm/models/swin_transformer.py", line 204, in forward
    attn = attn + self._get_rel_pos_bias()
 |   File "/home/rxchi1d/anaconda3/envs/ML2/lib/python3.9/site-packages/timm/models/swin_transformer.py", line 310, in forward
    attn_windows = self.attn(x_windows, mask=self.attn_mask)  # num_win*B, window_size*window_size, C
 |   File "/home/rxchi1d/anaconda3/envs/ML2/lib/python3.9/site-packages/timm/models/swin_t